# ML-06 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/pr120107/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

# Clone the repo if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Repo contents:", os.listdir()[:10])

Working directory: /content/flyrank-ml-internship
Repo contents: ['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs']


In [ ]:
import os

data_path = "data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(data_path))
print("Dataset path:", os.path.abspath(data_path))

Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total pages:", len(df))


# ---------------------------------------------------------
# Signal 1: Staleness
# ---------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, 364, np.inf],
    labels=["<90 days", "90-179 days", "180-364 days", "365+ days"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("\nSignal 1: Days since last update")
print(staleness_table.to_string(index=False))


# ---------------------------------------------------------
# Signal 2: Engagement
# ---------------------------------------------------------

df["engagement_bucket"] = pd.cut(
    df["engagement_rate"],
    bins=[-np.inf, 0.30, 0.60, np.inf],
    labels=["<30%", "30-60%", ">60%"]
)

engagement_table = (
    df.groupby("engagement_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

print("\nSignal 2: Engagement rate")
print(engagement_table.to_string(index=False))

Total pages: 30000

Signal 1: Days since last update
staleness_bucket     n  median_impressions
        <90 days 20655               472.0
     90-179 days  9171              1692.0
    180-364 days   169                16.0
       365+ days     5                 2.0

Signal 2: Engagement rate
engagement_bucket     n  median_impressions
             <30% 21659               322.0
           30-60%   130             10851.0
             >60%  8211              5523.0


## Signal verdicts

**Staleness - MIXED**

The data shows that most pages are less than 180 days since their last update, while only 174 pages are at least 180 days old. The stale groups also have much lower median impressions than the newer groups. This means staleness alone is not a strong indicator of search visibility in this dataset, but it may still be useful when combined with another signal.

**Engagement rate - CONFIRMED**

The engagement buckets show a clear directional difference in median impressions. Pages with engagement above 60% have a median of 5,523 impressions, compared with 322 for pages below 30%. This suggests engagement rate is a useful signal for distinguishing different page-performance patterns.

These are observed relationships in the starter dataset, not causal findings.

## 2. Build the ranked queue (writes the CSV)

### My baseline rule

I will prioritize pages that have relatively high engagement and enough search visibility to be useful for review.

The score combines engagement rate and search impressions. Pages with stronger engagement and higher visibility receive a higher score.

The reason code is `high_engagement_visible` and the action label is `review_archetype`.

This is a decision-support baseline. A high score does not mean that the page is guaranteed to perform better or that engagement causes higher visibility.

In [ ]:
# ---------------------------------------------------------
# Build the baseline ranked queue
# ---------------------------------------------------------

import os

# Make sure the required output folder exists
os.makedirs("work/outputs", exist_ok=True)

# Normalize engagement rate to a 0-1 range
engagement_score = (
    df["engagement_rate"]
    .clip(lower=0)
    .fillna(0)
)

# Normalize log impressions so very large pages
# do not completely dominate the score
visibility_score = (
    np.log1p(df["impressions_90d"].clip(lower=0))
    / np.log1p(df["impressions_90d"].max())
)

# Combine the two observed signals
df["baseline_score"] = (
    100 * (
        0.60 * engagement_score +
        0.40 * visibility_score
    )
)

# One reason code
df["reason_code"] = np.where(
    (df["engagement_rate"] >= 0.60) &
    (df["impressions_90d"] >= 500),
    "high_engagement_visible",
    "other"
)

# One action label
df["action"] = np.where(
    df["reason_code"] == "high_engagement_visible",
    "review_archetype",
    "monitor"
)

# Rank highest scoring pages first
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Write the required ranked queue
output_path = "work/outputs/baseline_action_score.csv"

df[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "engagement_rate",
        "impressions_90d"
    ]
].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows written:", len(df))

Saved: work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Show the top 10 pages for manual review

top10 = df.head(10)

top10[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "engagement_rate",
        "impressions_90d"
    ]
]

,rank,content_id,baseline_score,reason_code,action,engagement_rate,impressions_90d
0,1,content_139f09ffd77b,6026.627493,high_engagement_visible,review_archetype,100.0,6364
1,2,content_d016173fedb8,6024.362889,high_engagement_visible,review_archetype,100.0,3021
2,3,content_8aa0d6cfc237,6024.207055,high_engagement_visible,review_archetype,100.0,2870
3,4,content_e9785b5bd320,6022.832946,high_engagement_visible,review_archetype,100.0,1826
4,5,content_4c8b6a410f88,6022.677618,high_engagement_visible,review_archetype,100.0,1735
5,6,content_5f81f901e2e6,6022.344826,high_engagement_visible,review_archetype,100.0,1555
6,7,content_64b2508ad098,6022.297569,high_engagement_visible,review_archetype,100.0,1531
7,8,content_ce611830d125,6022.045283,high_engagement_visible,review_archetype,100.0,1409
8,9,content_e7b4e9967ffa,6021.979900,high_engagement_visible,review_archetype,100.0,1379
9,10,content_7a08dc784d80,6021.872273,high_engagement_visible,review_archetype,100.0,1331


## Top-10 review

1. **Rank 1:** Action: review_archetype. It has 100% engagement and 6,364 impressions, so it ranks highly under the baseline rule. It could be wrong if the high engagement rate is based on limited or unusual user activity.

2. **Rank 2:** Action: review_archetype. It has 100% engagement and 3,021 impressions, giving it strong visibility and engagement signals. It could be wrong if the engagement measurement does not represent typical page performance.

3. **Rank 3:** Action: review_archetype. It has 100% engagement and 2,870 impressions. It could be wrong if the high engagement rate is caused by a small or unusual group of sessions.

4. **Rank 4:** Action: review_archetype. It has 100% engagement and 1,826 impressions. It could be wrong if the engagement signal is not representative of normal traffic.

5. **Rank 5:** Action: review_archetype. It has 100% engagement and 1,735 impressions. It could be wrong if the observed engagement is temporary or noisy.

6. **Rank 6:** Action: review_archetype. It has 100% engagement and 1,555 impressions. It could be wrong if the engagement rate does not reflect the page's broader performance.

7. **Rank 7:** Action: review_archetype. It has 100% engagement and 1,531 impressions. It could be wrong if the engagement measurement is affected by limited traffic or unusual behavior.

8. **Rank 8:** Action: review_archetype. It has 100% engagement and 1,409 impressions. It could be wrong if the high engagement rate is not stable over time.

9. **Rank 9:** Action: review_archetype. It has 100% engagement and 1,379 impressions. It could be wrong if the observed engagement is not representative of future performance.

10. **Rank 10:** Action: review_archetype. It has 100% engagement and 1,331 impressions. It could be wrong if the engagement signal is temporary or affected by limited traffic.

These are ranked for review based on observed signals. The score does not prove that any page needs a specific content change.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Inspect some of the weakest-ranked pages

bottom10 = df.tail(10)

bottom10[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "engagement_rate",
        "impressions_90d"
    ]
]

,rank,content_id,baseline_score,reason_code,action,engagement_rate,impressions_90d
29990,29991,content_92ceb4aee549,2.107282,other,monitor,0.0,1
29991,29992,content_68247efaf942,2.107282,other,monitor,0.0,1
29992,29993,content_4480ae5d74f1,2.107282,other,monitor,0.0,1
29993,29994,content_56b9158e1337,2.107282,other,monitor,0.0,1
29994,29995,content_c7fc8386f0aa,2.107282,other,monitor,0.0,1
29995,29996,content_04e0b17d440e,2.107282,other,monitor,0.0,1
29996,29997,content_43921eb4e371,2.107282,other,monitor,0.0,1
29997,29998,content_31c8f34527e2,2.107282,other,monitor,0.0,1
29998,29999,content_ac94cdeeb855,2.107282,other,monitor,0.0,1
29999,30000,content_b114f27e79c9,2.107282,other,monitor,0.0,1


### Weak picks and leakage check

The weakest-ranked pages all had 0% engagement and only 1 impression. They appear to be reasonable low-priority candidates because the baseline is intended to prioritize pages with stronger engagement and meaningful visibility.

However, this also shows a limitation of the baseline: pages with extremely high engagement can dominate the ranking, while very low-volume pages are pushed to the bottom. A future version should test minimum-volume thresholds and compare engagement within more appropriate groups.

### Leakage check

The baseline uses `engagement_rate` and `impressions_90d`, which are observed signals from the available dataset. It does not use future-window metrics, the starter label, product decision flags, or model predictions.

This baseline is therefore intended as a simple decision-support ranking, not a prediction of future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.